In [18]:
from andromeda.config import AndromedaConfig, AgentConfig, SupervisorConfig, ModelConfig, PlannerConfig
from andromeda.core.team import Team
from andromeda.tools.filesystem import make_filesystem_tools

# Define model once
model_cfg = ModelConfig(name="llama3.2:3b", provider="ollama")

# Create filesystem tools (make_filesystem_tools returns dict, convert to list)
fs_tools_dict = make_filesystem_tools(allowed_dirs=["."])
fs_tools_list = list(fs_tools_dict.values())

# Agents with tools and stop conditions
research_agent = AgentConfig(
    name="researcher",
    model=model_cfg,
    tools=fs_tools_list,
    prompt="""You are a research agent. Your task is to research a topic and provide a concise summary.
When you have gathered the information and provided a clear response, STOP.
Do not ask for more information or iterate endlessly.
Focus on accuracy and clarity. Complete your response and await next instruction."""
)

writer_agent = AgentConfig(
    name="writer_agent",
    model=model_cfg,
    tools=fs_tools_list,
    prompt="""You are a writer agent. Your task is to write the final response back into te file as per instructed by the user
    You can use tools and its variouse functions to write the the specified file."""
)

# Supervisor with routing prompt and stop condition
supervisor = SupervisorConfig(
    name="supervisor",
    model=model_cfg,
    prompt="""You are a supervisor. Route the user's question to the appropriate agent (researcher).
Wait for their response. Once the agent provides findings, return the result to the user.
Do not iterate endlessly. Complete the task when the agent has responded."""
)

# Planner for decomposition
planner = PlannerConfig(
    model=model_cfg,
    task_type="research",
)

# Create config with all required components
config = AndromedaConfig(
    agents={"researcher": research_agent,"writer":writer_agent},
    supervisor=supervisor,
    planner=planner
)

# Create team and execute
team = Team(config)
print("Starting team execution...")
result = team.begin("Summarize the impact AI has on modern logistics in 2 sentences. Write a code to add two numbers in python and write down code back in Readme file")
print("\nResult:")
if isinstance(result, dict) and "messages" in result:
    print(result["messages"][-1].content)
else:
    print(result)

2026-07-16T08:16:10 - INFO - [ANDROMEDA] ├── Team.__init__ called...
2026-07-16T08:16:10 - INFO - [ANDROMEDA] │    ├─── dedent called...
2026-07-16T08:16:10 - INFO - [ANDROMEDA] │    ├─── dedent Ok. (took 0.00005 seconds)
2026-07-16T08:16:10 - INFO - [ANDROMEDA] │    ├─── dedent called...
2026-07-16T08:16:10 - INFO - [ANDROMEDA] │    ├─── dedent Ok. (took 0.00004 seconds)
2026-07-16T08:16:10 - INFO - [ANDROMEDA] │    ├─── Team._build_workflow called...
2026-07-16T08:16:10 - INFO - [ANDROMEDA] │    ├─── Team._build_workflow Ok. (took 0.00012 seconds)
Starting team execution...2026-07-16T08:16:10 - INFO - [ANDROMEDA] ├── Team.__init__ Ok. (took 0.18525 seconds)

2026-07-16T08:16:10 - INFO - [ANDROMEDA] ├── Team.begin called...
2026-07-16T08:16:10 - INFO - [ANDROMEDA] │    ├─── Team._planner_step called...
2026-07-16T08:17:32 - INFO - [ANDROMEDA] │    │    ├─── Team._merge_state called...
2026-07-16T08:17:32 - INFO - [ANDROMEDA] │    │    ├─── Team._merge_state Ok. (took 0.00025 seconds)


In [15]:
from andromeda.core.agent import Agent
from andromeda.config import AgentConfig, ModelConfig
from andromeda.tools.filesystem import make_filesystem_tools
from andromeda import HumanMessage

fs_tools = make_filesystem_tools(["/home/rakeshchanda/TrainingFolder/CodingLive/"])

agent = Agent(
  AgentConfig(
    name="file_agent",
    model=ModelConfig(name="llama3.2:3b", provider="ollama"),
    tools=list(fs_tools.values()),
    prompt="Use filesystem tools carefully.",
  )
)

response = agent.invoke([
  HumanMessage(content="List files in the workspace and read README.md")
])
print(response[-1].content)

To list files in the workspace, you can use the `ls` command.

Here's how to do it:

```bash
ls /workspace
```

This will display all files and directories present in the `/workspace` directory.

Next, I'll read the contents of README.md:

```bash
cat /workspace/README.md
```


In [10]:
from andromeda.tools.filesystem import make_filesystem_tools

fs_tools = make_filesystem_tools(["/home/rakeshchanda/TrainingFolder/CodingLive"])

print(fs_tools["list_directory"].invoke({"path": "./."}))
print("="*40)
print(fs_tools["read_file"].invoke({"path": "README.md", "start_line": 0, "end_line": 40}))
print("="*40)

edit_result = fs_tools["search_and_replace_file_edit"].invoke(
  {
    "path": "README.md",
    "search": "This repo is created for training purposes.",
    "replace": "This repo is newly created for training purposes.",
  }
)
print(edit_result)

[FILE] day01.py
[FILE] research_agent.py
[FILE] day02.ipynb
[FILE] README.md
[DIR] AI_Code_Reviewer_Agent
[FILE] day03.ipynb
[DIR] .git
[DIR] Data
[FILE] pyproject.toml
[DIR] vector_embedding_db
[DIR] AI_SDLC
[DIR] andromeda.egg-info
[DIR] andromeda
[DIR] Andromeda_Agent_01
# Training
This repo is newly created for training purposes.
'This repo is created for training purposes.' not found in README.md


In [14]:
from andromeda.workspace import FileSeed, WorkspacePolicy, WorkspaceSession

session = WorkspaceSession.create(
    backend="local_fs",
    root="./workspace",
    seed=[FileSeed(path="AGENTS.md", content="Work only in this workspace.")],
    policy=WorkspacePolicy(read_only=False, enable_shell=True),
)

tools = session.tools()

In [19]:
from andromeda.core.agent import Agent
from andromeda.core.supervisor import Supervisor
from andromeda.config import AgentConfig, SupervisorConfig, ModelConfig
from andromeda import HumanMessage

# Specialist worker configs
researcher = AgentConfig(
    name="researcher",
    model=ModelConfig(name="llama3.2:3b", provider="ollama"),
    prompt="Focus on evidence gathering and source quality.",
)
writer = AgentConfig(
    name="writer",
    model=ModelConfig(name="llama3.2:3b", provider="ollama"),
    prompt="Produce clear, concise writing from verified notes.",
)

# Supervisor config
supervisor_cfg = SupervisorConfig(
    name="supervisor",
    model=ModelConfig(name="llama3.2:3b", provider="ollama"),
    prompt="Plan and coordinate agents to fully cover the task.",
    enable_planning=True,
)

supervisor = Supervisor(agents=[researcher, writer], config=supervisor_cfg)

state = {
    "messages": [HumanMessage(content="Research EV trends and draft a short brief.")],
    "plan": [],
}
result = supervisor.supervise(state)
print(result["messages"][-1].content)

**Answer:**

Electric Vehicle (EV) Market Trends Analysis

The global electric vehicle market has experienced rapid growth over the past decade, driven by increasing concerns about climate change, air pollution, and energy security.

**Key Drivers:**

1.  **Government Incentives**: Governments worldwide have implemented policies to encourage the adoption of electric vehicles, including tax incentives and investment in EV infrastructure.
2.  **Technological Advancements**: Improvements in battery technology, such as solid-state batteries and thermoelectric converters, are driving down costs and improving range, making EVs more competitive with traditional vehicles.
3.  **Growing Consumer Awareness**: Increasing awareness about climate change and environmental concerns is driving consumer demand for eco-friendly vehicles like EVs.

**Challenges:**

1.  **Higher Upfront Costs**: While costs are decreasing over time, EVs remain more expensive than traditional vehicles, making them less acc